# AR-SSL 肺结节下游微调（NoduleMNIST3D）

本笔记本在本地 **MedMNIST `nodulemnist3d.npz`** 上对已预训练权重做 **二分类微调**，与仓库 `downstream/nodule/train_and_eval.py` 流程一致。

**数据**：`*.npz` 须放在某一目录下，文件名须为 **`nodulemnist3d.npz`**（与本机路径相符即可）。下游默认 Zenodo **28³**；若与你的预训练 **128³、patch 16** 不一致，会自动三线性插值到 **128³**。

**权重**：支持你提供的单文件 checkpoint；若为 `ReconModel` 保存的字典且键名以 **`model.`** 开头，将只加载骨干，分类头随机初始化。

**依赖**（按需安装）：`medmnist` `torch` `torchio` `tensorboardX` `tqdm` `monai` `timm` `transformers`

按从上到下顺序依次运行。

In [1]:
from __future__ import annotations

from pathlib import Path

# ——用户路径——
NPZ_PATH = Path(r"D:\finetune\10519652\nodulemnist3d.npz")
PRETRAIN_CKPT = Path(r"D:\finetune\ar-ssl4m\ar-ssl4m\checkpoints\0\0.pth")
REPO_ROOT = Path(r"D:\Cursor\AR-SSL4M-DEMO").resolve()
OUTPUT_ROOT = Path(r"D:\finetune\ar-ssl4m\nodule_finetune_outputs")

# MedMNIST 要求 root 目录下存在 nodulemnist3d.npz
DATA_ROOT = NPZ_PATH.parent
assert NPZ_PATH.name == "nodulemnist3d.npz", f"文件名应为 nodulemnist3d.npz，当前: {NPZ_PATH.name}"
assert NPZ_PATH.is_file(), f"找不到数据: {NPZ_PATH}"
assert PRETRAIN_CKPT.is_file(), f"找不到权重: {PRETRAIN_CKPT}"
assert (REPO_ROOT / "downstream" / "nodule").is_dir(), f"REPO_ROOT 无效: {REPO_ROOT}"

IMG_SIZE = 128
PATCH_SIZE = 16
BATCH_SIZE = 16
NUM_EPOCHS = 100
LR = 1.5e-5
GPU_IDS = "0"   # CPU: 设为 "" 且在下方 device 中会退化为 cpu
RANDOM_SEED = 42
RUN_NAME = "ar_ssl_ft"

print("DATA_ROOT:", DATA_ROOT)
print("PRETRAIN_CKPT:", PRETRAIN_CKPT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)

DATA_ROOT: D:\finetune\10519652
PRETRAIN_CKPT: D:\finetune\ar-ssl4m\ar-ssl4m\checkpoints\1\1.pth
OUTPUT_ROOT: D:\finetune\ar-ssl4m\nodule_finetune_outputs


In [2]:
import sys
import random
import time
import warnings
from copy import deepcopy

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
import os

NODULE_DIR = REPO_ROOT / "downstream" / "nodule"
sys.path.insert(0, str(NODULE_DIR))

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8"

# 与 train_and_eval 一致的非确定性设置（若报错可注释 use_deterministic）
warnings.filterwarnings("ignore", category=UserWarning)
torch.backends.cudnn.benchmark = False
try:
    torch.use_deterministic_algorithms(True)
except Exception as e:
    print("跳过 torch.use_deterministic_algorithms:", e)

torch.manual_seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

import medmnist
from medmnist import INFO, Evaluator
from tqdm import trange
from tensorboardX import SummaryWriter

from utils import Transform3D
from base_model import BaseModel, ClassificationModel
import train_and_eval as te

# train() 模块内使用 global iteration，首次训练前赋值
te.iteration = 0

print("sys.path head:", sys.path[:3])

c:\ProgramData\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


sys.path head: ['D:\\Cursor\\AR-SSL4M-DEMO\\downstream\\nodule', 'c:\\ProgramData\\miniconda3\\python313.zip', 'c:\\ProgramData\\miniconda3\\DLLs']


In [3]:
import torch.nn.functional as F

def peek_spatial_from_npz(path: Path):
    """查看 train_images 单样本空间尺寸 (D,H,W)。"""
    z = np.load(path, mmap_mode="r")
    x = z["train_images"]
    # MedMNIST: (N, 28, 28, 28) uint8
    while x.ndim > 3:
        x = x[0]
    return int(x.shape[0]), int(x.shape[1]), int(x.shape[2])


_sp = peek_spatial_from_npz(NPZ_PATH)
_need_resize = _sp != (IMG_SIZE, IMG_SIZE, IMG_SIZE)
print("当前 npz train 单体素形状 (D,H,W):", _sp, "→ resize 到", IMG_SIZE, "?", _need_resize)


class Transform3DFinetune(Transform3D):
    """在原有 transpose + 可选翻转前，按需将 (C,D,H,W) 插值到 IMG_SIZE³。"""

    def __init__(self, train_augmentation: bool = False, resize_to=None):
        super().__init__(train_augmentation=train_augmentation)
        self.resize_to = resize_to

    def __call__(self, voxel: np.ndarray) -> np.ndarray:
        if self.resize_to is not None:
            t = torch.from_numpy(np.asarray(voxel, dtype=np.float32)).unsqueeze(0)
            # (1, C, D, H, W)
            sz = (self.resize_to, self.resize_to, self.resize_to)
            t = F.interpolate(t, size=sz, mode="trilinear", align_corners=False)
            voxel = t.squeeze(0).numpy()
        return super().__call__(voxel)


_r = IMG_SIZE if _need_resize else None
train_transform = Transform3DFinetune(train_augmentation=True, resize_to=_r)
eval_transform = Transform3DFinetune(train_augmentation=False, resize_to=_r)

当前 npz train 单体素形状 (D,H,W): (28, 28, 28) → resize 到 128 ? True


In [4]:
data_flag = "nodulemnist3d"
info = INFO[data_flag]
n_classes = len(info["label"])
DataClass = getattr(medmnist, info["python_class"])

train_ds = DataClass(split="train", transform=train_transform, download=False, as_rgb=False, root=str(DATA_ROOT))
train_ds_eval = DataClass(split="train", transform=eval_transform, download=False, as_rgb=False, root=str(DATA_ROOT))
val_ds = DataClass(split="val", transform=eval_transform, download=False, as_rgb=False, root=str(DATA_ROOT))
test_ds = DataClass(split="test", transform=eval_transform, download=False, as_rgb=False, root=str(DATA_ROOT))

train_loader = data.DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
train_loader_eval = data.DataLoader(train_ds_eval, batch_size=BATCH_SIZE, shuffle=False)
val_loader = data.DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = data.DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

train_evaluator = Evaluator(data_flag, "train", root=str(DATA_ROOT))
val_evaluator = Evaluator(data_flag, "val", root=str(DATA_ROOT))
test_evaluator = Evaluator(data_flag, "test", root=str(DATA_ROOT))

xb, yb = next(iter(train_loader))
print("batch x shape", tuple(xb.shape), "y", tuple(yb.shape), "classes", n_classes)

batch x shape (16, 1, 128, 128, 128) y (16, 1) classes 2


In [5]:
def pick_state_dict(blob: dict) -> dict:
    """兼容 train_and_eval 期望的 {"state_dict": ...} 与裸权重 dict。"""
    if not isinstance(blob, dict):
        raise TypeError("checkpoint 应为 dict")
    if "state_dict" in blob and isinstance(blob["state_dict"], dict):
        return blob["state_dict"]
    # 形如 ReconModel 整体 state_dict
    if any(k.startswith("model.") for k in blob.keys()) or any(k.startswith("decoder_pred.") for k in blob.keys()):
        return blob
    return blob


def strip_prefix(keys_state: dict) -> dict:
    out = {}
    for k, v in keys_state.items():
        nk = k
        if nk.startswith("module."):
            nk = nk[len("module.") :]
        if nk.startswith("_orig_mod."):
            nk = nk[len("_orig_mod.") :]
        out[nk] = v
    return out


# device
_gids = [int(x) for x in GPU_IDS.split(",") if x.strip() != "" and int(x) >= 0]
if _gids and torch.cuda.is_available():
    device = torch.device(f"cuda:{_gids[0]}")
else:
    device = torch.device("cpu")
print("device:", device)

class Model_Config:
    pos_type = "sincos3d"
    norm_pixel_loss = True


_cfg = Model_Config()
_cfg.img_size = [IMG_SIZE, IMG_SIZE, IMG_SIZE]
_cfg.patch_size = [PATCH_SIZE, PATCH_SIZE, PATCH_SIZE]
_cfg.hidden_size = 768
_cfg.intermediate_size = 3072
_cfg.num_attention_heads = 12
_cfg.num_key_value_heads = 12
_cfg.num_hidden_layers = 12

_encoder = BaseModel(_cfg)
model = ClassificationModel(_encoder, n_classes)

_raw = torch.load(PRETRAIN_CKPT, map_location="cpu")
_sd = strip_prefix(pick_state_dict(_raw))
_enc_only = {k: v for k, v in _sd.items() if k.startswith("model.")}
_incomp = model.load_state_dict(_enc_only, strict=False)
print("已加载前缀 'model.' 的张量:", len(_enc_only))
print("missing keys (示例前 8):", _incomp.missing_keys[:8])
print("unexpected keys (示例前 8):", _incomp.unexpected_keys[:8])

model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-2)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

device: cuda:0
已加载前缀 'model.' 的张量: 161
missing keys (示例前 8): ['score.weight']
unexpected keys (示例前 8): []


In [6]:
out_dir = OUTPUT_ROOT / data_flag / time.strftime("%y%m%d_%H%M%S")
out_dir.mkdir(parents=True, exist_ok=True)
writer = SummaryWriter(log_dir=str(out_dir / "tensorboard"))
print("输出目录:", out_dir)

patience = 30
best_auc, best_acc = 0.0, 0.0
best_epoch = 0
best_model = deepcopy(model)
te.iteration = 0

for epoch in trange(NUM_EPOCHS):
    if epoch - best_epoch > patience:
        print("early stop")
        break

    tr_loss = te.train(model, train_loader, criterion, optimizer, device, writer)
    val_m = te.test(model, val_evaluator, val_loader, criterion, device, RUN_NAME)

    writer.add_scalar("val_loss", val_m[0], epoch)
    writer.add_scalar("val_auc", val_m[1], epoch)
    writer.add_scalar("val_acc", val_m[2], epoch)
    writer.add_scalar("train_epoch_loss", tr_loss, epoch)

    cur_auc, cur_acc = val_m[1], val_m[2]
    if cur_auc > best_auc:
        best_epoch = epoch
        best_auc = cur_auc
        best_acc = cur_acc
        best_model = deepcopy(model)
        print("best @", epoch, "auc", cur_auc, "acc", cur_acc)

    scheduler.step()

_best_path = out_dir / "best_model.pth"
torch.save({"net": best_model.state_dict()}, _best_path)
print("saved:", _best_path)

train_m = te.test(best_model, train_evaluator, train_loader_eval, criterion, device, RUN_NAME, str(out_dir))
val_m = te.test(best_model, val_evaluator, val_loader, criterion, device, RUN_NAME, str(out_dir))
test_m = te.test(best_model, test_evaluator, test_loader, criterion, device, RUN_NAME, str(out_dir))

log_txt = (
    f"{data_flag}\n"
    f"train  auc: {train_m[1]:.5f}  acc: {train_m[2]:.5f}\n"
    f"val    auc: {val_m[1]:.5f}  acc: {val_m[2]:.5f}\n"
    f"test   auc: {test_m[1]:.5f}  acc: {test_m[2]:.5f}\n"
)
print(log_txt)
(out_dir / f"{data_flag}_log.txt").write_text(log_txt, encoding="utf-8")
writer.close()

输出目录: D:\finetune\ar-ssl4m\nodule_finetune_outputs\nodulemnist3d\260525_235250


  1%|          | 1/100 [00:19<32:35, 19.75s/it]

best @ 0 auc 0.8199767711962834 acc 0.793939393939394


  2%|▏         | 2/100 [00:39<31:56, 19.56s/it]

best @ 1 auc 0.8364305071622145 acc 0.8424242424242424


...

best @ 4 auc 0.8935346496322106 acc 0.8545454545454545


 12%|█▏        | 12/100 [03:53<28:27, 19.41s/it]

best @ 11 auc 0.9076655052264808 acc 0.8484848484848485


 42%|████▏     | 42/100 [13:40<18:52, 19.52s/it]


early stop
saved: D:\finetune\ar-ssl4m\nodule_finetune_outputs\nodulemnist3d\260525_235250\best_model.pth
nodulemnist3d
train  auc: 0.91527  acc: 0.87565
val    auc: 0.90767  acc: 0.84848
test   auc: 0.92149  acc: 0.83871

